<a href="https://colab.research.google.com/github/fanwenlin/TDDC17-lab/blob/main/lab6/planning_lab_stub.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Planning Lab

In [ ]:
!pip install unified-planning==1.1.0
!pip install up_fast_downward==0.4.1

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.6/706.6 kB 6.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 12.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.4/49.4 MB 8.1 MB/s eta 0:00:00


In [ ]:
!pip install matplotlib==3.7.1  # for visualisation in this notebook

In [ ]:
from unified_planning.shortcuts import *

import unified_planning as up

up.shortcuts.get_environment().credits_stream = None

## Part (a): Model the Task

The code below models a simpler version of the Household Robot domain with two rooms and one open door between them. The robot is in room A initially and needs to move to room B. You can use it as a starting point for your own solution to the full Household Robot task.

In [ ]:

def get_planning_task():
    # Declare user types.
    Room = UserType("Room")
    Door = UserType("Door")

    # Declare predicates.
    robot_in = up.model.Fluent("robot_in", BoolType(), r=Room)
    connected = up.model.Fluent("connected", BoolType(), r1=Room, d=Door, r2=Room)
    room_has_door = up.model.Fluent("room_has_door", BoolType(), r=Room, d=Door)

    # Add (typed) objects to problem.
    problem = up.model.Problem("household")

    def get_room(room):
        return up.model.Object(f"room{room}", Room)

    def get_door(door):
        return up.model.Object(f"door{door}", Door)

    roomA = get_room("A")
    roomB = get_room("B")
    rooms = [roomA, roomB]

    doorB = get_door("B")
    doors = [doorB]

    problem.add_objects(rooms)
    problem.add_objects(doors)

    connections = [
        (roomA, doorB, roomB)
    ]

    rooms_and_doors = []
    for room1, door, room2 in connections:
        rooms_and_doors.append((room1, door))
        rooms_and_doors.append((room2, door))

    # Specify the initial state.
    problem.add_fluent(robot_in, default_initial_value=False)
    problem.add_fluent(connected, default_initial_value=False)
    problem.add_fluent(room_has_door, default_initial_value=False)
    problem.set_initial_value(robot_in(roomA), True)
    for room1, door, room2 in connections:
        problem.set_initial_value(connected(room1, door, room2), True)
        problem.set_initial_value(connected(room2, door, room1), True)
    for room, door in rooms_and_doors:
        problem.set_initial_value(room_has_door(room, door), True)

    # Add actions.
    move = up.model.InstantaneousAction("move", room1=Room, door=Door, room2=Room)
    room1 = move.parameter("room1")
    door = move.parameter("door")
    room2 = move.parameter("room2")
    move.add_precondition(robot_in(room1))
    move.add_precondition(connected(room1, door, room2))
    move.add_effect(robot_in(room1), False)
    move.add_effect(robot_in(room2), True)
    problem.add_action(move)

    # Specify the goal.
    problem.add_goal(robot_in(roomB))

    # We want to minimize the plan cost.
    problem.add_quality_metric(MinimizeActionCosts({}, default=Int(1)))
    return problem

problem = get_planning_task()


## Part (b): Find a (possibly suboptimal) plan

Solve the task with greedy best-first search using the FF heuristic. The example code below uses the h^add heuristic. You need to inspect the output to stdout to see the heuristic value of the initial state.

In [ ]:
params = {
    "fast_downward_search_config": "eager_greedy([add()])"
}

with OneshotPlanner(name="fast-downward", params=params) as planner:
    result = planner.solve(problem)
    if result.status == up.engines.PlanGenerationResultStatus.SOLVED_SATISFICING:
        print("Found a plan of length:", len(result.plan.actions))
        print(result.plan)
        with PlanValidator() as validator:
            val_result = validator.validate(problem, result.plan)
            print("Plan cost:", val_result.metric_evaluations)
    else:
        print("No plan found.")

Found a plan of length: 1
SequentialPlan:
    move(roomA, doorB, roomB)
Plan cost: {minimize actions-cost: {'default': 1}: 1}


## Part (c): Find an optimal plan

Solve the task with A* using the `iPDB` heuristic.